# 4.3 ACLNN GEMM

## 本节学习目标

- 掌握 Tensor/Workspace/Executor 流程
- 验证 GEMM 结果

## 环境检查

直接检查本节需要的运行环境；若检查失败，请先在对应 CPU/NPU 节点加载课程要求的工具链。


In [ ]:
%%bash
set -e
command -v cmake
command -v npu-smi
npu-smi info
printf "ASCEND_HOME_PATH=%s\n" "${ASCEND_HOME_PATH:?请先 source CANN set_env.sh}"


## 调用流程

工程创建 A、B、C、output 四个 aclTensor。每次执行调用 `aclnnGemmGetWorkspaceSize` 获取 workspace size 和 executor，分配 workspace，再调用 `aclnnGemm` 并同步 stream。

## 正确性

Host 固定随机种子生成 FP32 A/B，以三重循环计算 reference。`cubeMathType=0` 保持 FP32，最后计算相对 L2 误差。

## 构建并运行

从当前章节目录执行下面的 Cell，并对照随后给出的检查点阅读输出。

In [ ]:
!cd src/acl_operator_calls/GEMM-acl && bash scripts/build.sh
!cd src/acl_operator_calls/GEMM-acl && bash scripts/run.sh --m 1024 --k 1024 --n 1024 --warmup 10 --repeat 100

## 预期现象与结果分析

真实 CANN 环境应输出平均时间、M/K/N 和误差。无 CANN 时工程可能构建诊断 stub；stub 返回失败且不代表算子结果。

## 原工程历史参考输出

`acl-c/README.md` 记录了 CANN 9.0.0、固定随机种子、`M=K=N=1024`、`warmup=10`、`repeat=100` 的远程 Ascend 实测：平均 `0.056814 ms`，相对 L2 误差 `6.409110591e-07`。该记录用于确认 ACLNN 生命周期与正确性门槛；耗时不是课程规定值，不能脱离设备、CANN 和计时范围比较。

## 课后实践

将 GEMM 调用步骤按资源创建、算子准备、执行、同步、验证和释放分类。

参考答案见 `answer/04.03_answer.md`。

## 直接执行实验

该 Cell 强制真实 ACL/CANN 后端并独立运行 GEMM；配置失败时不会退回 Host Stub。


In [ ]:
%%bash
set -e
cd src/acl_operator_calls/GEMM-acl
cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DACL_C_STUB=OFF
cmake --build build -j
./build/bin/gemm_acl --m 512 --n 512 --k 512 --warmup 3 --repeat 10 --device 0
